In [0]:
%run ./00_config
%run ./00_functions

In [0]:
from pyspark.sql import functions as F

In [0]:
bronze_df = spark.read.table(bronze_table)
validate_df = validate_transaction(bronze_df)

valid_df = validate_df.filter(
    F.col('dq_status') == 'VALID'
)

invalid_df = validate_df.filter(
    F.col('dq_status') == 'INVALID')

valid_df.write.mode('overwrite').format('delta').saveAsTable(silver_table)
invalid_df.write.mode('overwrite').format('delta').saveAsTable(invalid_table)

In [0]:
bronze_rows = bronze_df.count()
valid_rows = valid_df.count()
invalid_rows = invalid_df.count()
assert bronze_rows == valid_rows + invalid_rows, "Valid and invalid rows do not sum to bronze rows"
assert valid_df.filter(F.col('dq_reason').isNotNull()).count() == 0, "Valid rows have non-null DQ reasons"
assert invalid_df.filter(F.col('dq_reason').isNull()).count() == 0, "Invalid rows have null DQ reasons"
